In [ ]:
# Lattice of Itemsets
# 這邊做的就是按照排列去找出所有成員的組成可能，接著把算真的 sets 裡面有出現幾次

In [3]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. 上課投影片的 5 筆交易
transactions = [
    ['bread', 'butter', 'milk', 'sugar'],          # T1
    ['butter', 'flour', 'milk', 'sugar'],          # T2
    ['butter', 'eggs', 'milk', 'salt'],            # T3
    ['eggs'],                                      # T4
    ['butter', 'flour', 'milk', 'salt', 'sugar'],  # T5
]

# 2. 轉成 one-hot 表（每個 item 一欄，True/False）
te = TransactionEncoder()
df = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)
print("One-hot 交易表：")
print(df)

# 3. Apriori：找 frequent itemsets
#    min_support=0.6 等於 5 筆裡至少出現 3 次（注意這裡是「比例」不是 count）
freq = apriori(df, min_support=0.6, use_colnames=True)
print("\nFrequent itemsets：")
print(freq)

One-hot 交易表：
   bread  butter   eggs  flour   milk   salt  sugar
0   True    True  False  False   True  False   True
1  False    True  False   True   True  False   True
2  False    True   True  False   True   True  False
3  False   False   True  False  False  False  False
4  False    True  False   True   True   True   True

Frequent itemsets：
   support                          itemsets
0      0.8               frozenset({butter})
1      0.8                 frozenset({milk})
2      0.6                frozenset({sugar})
3      0.8         frozenset({milk, butter})
4      0.6        frozenset({sugar, butter})
5      0.6          frozenset({milk, sugar})
6      0.6  frozenset({sugar, milk, butter})


In [4]:
# 4. 從 frequent itemsets 產生 association rules
#    篩 confidence ≥ 0.8
rules = association_rules(freq, metric="confidence", min_threshold=0.8)
print("\nAssociation rules：")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])


Association rules：
                  antecedents                consequents  support  confidence  \
0           frozenset({milk})        frozenset({butter})      0.8         1.0   
1         frozenset({butter})          frozenset({milk})      0.8         1.0   
2          frozenset({sugar})        frozenset({butter})      0.6         1.0   
3          frozenset({sugar})          frozenset({milk})      0.6         1.0   
4  frozenset({sugar, butter})          frozenset({milk})      0.6         1.0   
5    frozenset({milk, sugar})        frozenset({butter})      0.6         1.0   
6          frozenset({sugar})  frozenset({milk, butter})      0.6         1.0   

   lift  
0  1.25  
1  1.25  
2  1.25  
3  1.25  
4  1.25  
5  1.25  
6  1.25  


In [ ]:
# 完整 interestingness 指標
# lift 和 conviction 是 mlxtend 預設就會算的；Jaccard 自己加
rules['jaccard'] = rules['support'] / (
    rules['antecedent support'] + rules['consequent support'] - rules['support']
)

print("\nAssociation rules with interestingness measures：")
print(rules[[
    'antecedents', 'consequents',
    'support', 'confidence',
    'lift', 'jaccard', 'conviction'
]])


Association rules with interestingness measures：
                  antecedents                consequents  support  confidence  \
0           frozenset({milk})        frozenset({butter})      0.8         1.0   
1         frozenset({butter})          frozenset({milk})      0.8         1.0   
2          frozenset({sugar})        frozenset({butter})      0.6         1.0   
3          frozenset({sugar})          frozenset({milk})      0.6         1.0   
4  frozenset({sugar, butter})          frozenset({milk})      0.6         1.0   
5    frozenset({milk, sugar})        frozenset({butter})      0.6         1.0   
6          frozenset({sugar})  frozenset({milk, butter})      0.6         1.0   

   lift  jaccard  conviction  
0  1.25     1.00         inf  
1  1.25     1.00         inf  
2  1.25     0.75         inf  
3  1.25     0.75         inf  
4  1.25     0.75         inf  
5  1.25     0.75         inf  
6  1.25     0.75         inf  
